# Causal Flow SQLite Build

This notebook flattens `runs.bson` and `runs_ablation_minimality.bson` into analysis-friendly SQLite tables. By default it includes ablation runs and marks them with ablation metadata.

In [1]:
import sqlite3
import importlib
import build_sqlite
importlib.reload(build_sqlite)

from build_sqlite import DB_PATH, load_database, table_counts

In [2]:
# %load_ext autoreload
# %autoreload 2

In [3]:
conn, load_summary = load_database(include_ablations=True)

## Tables

- `runs`: one row per experiment run
- `traces`: one row per problem attempt, passing or failing
- `steps`: one row per agent trace step
- `repair_attempts`: one row per successful repairable step currently recoverable from the BSON
- `judge_votes`: one row per individual LLM judge vote
- `consensus_steps`: one row per step accepted by the multi-agent critique/council
- `trace_metrics`: one row per failed trace with attribution, repair, minimality, and critique summary metrics
- `triage_labels`: one row per failed trace for the Trace Triage paper labels; CausalFlow-repaired traces are auto-labeled `LOCAL_REPAIR`

In [4]:
table_counts(conn)

{'runs': 24,
 'traces': 3893,
 'steps': 41615,
 'repair_attempts': 2522,
 'judge_votes': 2624,
 'consensus_steps': 2512,
 'trace_metrics': 1739,
 'triage_labels': 1739}

In [5]:
for row in conn.execute("""
    SELECT benchmark,
           COUNT(*) AS traces,
           SUM(success = 1) AS passing,
           SUM(success = 0) AS failing,
           ROUND(AVG(success), 3) AS trace_accuracy
    FROM traces
    GROUP BY benchmark
    ORDER BY benchmark
"""):
    print(row)

('BrowseComp', 10, 3, 7, 0.3)
('GSM8K', 1494, 1244, 250, 0.833)
('Humaneval', 100, 99, 1, 0.99)
('MBPP', 1021, 549, 472, 0.538)
('MedBrowseComp', 485, 149, 336, 0.307)
('SealQA', 256, 110, 146, 0.43)
('gsm8k', 141, 0, 141, 0.0)
('mbpp', 200, 0, 200, 0.0)
('medbrowse', 134, 0, 134, 0.0)
('sealqa', 52, 0, 52, 0.0)


In [6]:
for row in conn.execute("""
    SELECT benchmark,
           COUNT(*) AS traces,
           SUM(success = 1) AS passing,
           SUM(success = 0) AS failing,
           ROUND(AVG(success), 3) AS trace_accuracy
    FROM traces
    GROUP BY benchmark
    ORDER BY benchmark
"""):
    print(row)

('BrowseComp', 10, 3, 7, 0.3)
('GSM8K', 1494, 1244, 250, 0.833)
('Humaneval', 100, 99, 1, 0.99)
('MBPP', 1021, 549, 472, 0.538)
('MedBrowseComp', 485, 149, 336, 0.307)
('SealQA', 256, 110, 146, 0.43)
('gsm8k', 141, 0, 141, 0.0)
('mbpp', 200, 0, 200, 0.0)
('medbrowse', 134, 0, 134, 0.0)
('sealqa', 52, 0, 52, 0.0)


In [7]:
for row in conn.execute("""
    SELECT judge_says_causal,
           is_repairable_step,
           COUNT(*) AS votes
    FROM judge_votes
    GROUP BY judge_says_causal, is_repairable_step
    ORDER BY judge_says_causal, is_repairable_step
"""):
    print(row)

(0, 0, 632)
(0, 1, 1588)
(1, 0, 178)
(1, 1, 226)


In [8]:
# Use this later if the database already exists and you do not want to rebuild it.
# conn = sqlite3.connect(DB_PATH)

In [9]:
for row in conn.execute("""
    SELECT COUNT(*) AS count
    FROM triage_labels
    WHERE needs_labeling = 1;
"""):
    print(row)

(703,)


In [11]:
from pathlib import Path
import json
import textwrap

OUTPUT_DIR = Path("data/labeling_exports")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MD_PATH = OUTPUT_DIR / "failed_traces.md"
JSONL_PATH = OUTPUT_DIR / "failed_traces.jsonl"

MAX_FIELD_CHARS = 1200
MAX_STEP_CHARS = 900


def clean_text(value, max_chars=None):
    if value is None:
        return ""
    text = str(value).replace("\r\n", "\n").replace("\r", "\n").strip()
    if max_chars and len(text) > max_chars:
        return text[:max_chars] + "\n...[truncated]"
    return text


def format_step(step):
    parts = [
        f"### Step {step['step_index']} | step_id={step['step_id']} | type={step['step_type']}"
    ]

    if step["tool_name"]:
        parts.append(f"- Tool: `{step['tool_name']}`")

    if step["tool_args_json"]:
        parts.append(f"- Tool args: `{clean_text(step['tool_args_json'], 500)}`")

    if step["text"]:
        parts.append("")
        parts.append(clean_text(step["text"], MAX_STEP_CHARS))

    if step["tool_output_json"]:
        parts.append("")
        parts.append("Tool output:")
        parts.append("```text")
        parts.append(clean_text(step["tool_output_json"], MAX_STEP_CHARS))
        parts.append("```")

    return "\n".join(parts)


trace_rows = conn.execute("""
    SELECT
        l.trace_id,
        l.problem_id,
        l.domain,
        l.benchmark,
        l.action_label,
        l.needs_labeling,
        l.is_local_repairable,
        l.num_successful_repair_steps,
        l.applicable_actions_json,
        t.problem_statement,
        t.gold_answer,
        t.final_answer,
        t.num_steps
    FROM triage_labels l
    JOIN traces t ON l.trace_id = t.trace_id
    WHERE l.is_ablation = 0 AND l.needs_labeling = 1
    ORDER BY l.domain, l.problem_id
""").fetchall()

columns = [desc[0] for desc in conn.execute("""
    SELECT
        l.trace_id,
        l.problem_id,
        l.domain,
        l.benchmark,
        l.action_label,
        l.needs_labeling,
        l.is_local_repairable,
        l.num_successful_repair_steps,
        l.applicable_actions_json,
        t.problem_statement,
        t.gold_answer,
        t.final_answer,
        t.num_steps
    FROM triage_labels l
    JOIN traces t ON l.trace_id = t.trace_id
    WHERE l.is_ablation = 0 AND l.needs_labeling = 1
    LIMIT 1
""").description]

trace_dicts = [dict(zip(columns, row)) for row in trace_rows]

with MD_PATH.open("w", encoding="utf-8") as md, JSONL_PATH.open("w", encoding="utf-8") as jsonl:
    md.write("# Failed Traces for Labeling\n\n")
    md.write(f"Total traces: {len(trace_dicts)}\n\n")
    md.write("Use this file to assign recovery labels:\n\n")
    md.write("```text\n")
    md.write("LOCAL_REPAIR, RETRY, REPLAN, RETRIEVE_MORE, TOOL_FIX, ESCALATE\n")
    md.write("```\n\n")

    for i, trace in enumerate(trace_dicts, start=1):
        steps = conn.execute("""
            SELECT
                step_uid,
                step_id,
                step_index,
                step_type,
                text,
                tool_name,
                tool_args_json,
                tool_output_json
            FROM steps
            WHERE trace_id = ?
            ORDER BY step_index
        """, (trace["trace_id"],)).fetchall()

        step_cols = [d[0] for d in conn.execute("""
            SELECT
                step_uid,
                step_id,
                step_index,
                step_type,
                text,
                tool_name,
                tool_args_json,
                tool_output_json
            FROM steps
            WHERE trace_id = ?
            LIMIT 1
        """, (trace["trace_id"],)).description]

        step_dicts = [dict(zip(step_cols, step)) for step in steps]

        record = dict(trace)
        record["steps"] = step_dicts
        jsonl.write(json.dumps(record, ensure_ascii=False) + "\n")

        md.write(f"\n---\n\n")
        md.write(f"## {i}. {trace['domain']} | {trace['problem_id']}\n\n")
        md.write(f"- `trace_id`: `{trace['trace_id']}`\n")
        md.write(f"- `benchmark`: `{trace['benchmark']}`\n")
        md.write(f"- `needs_labeling`: `{trace['needs_labeling']}`\n")
        md.write(f"- `current_action_label`: `{trace['action_label']}`\n")
        md.write(f"- `is_local_repairable`: `{trace['is_local_repairable']}`\n")
        md.write(f"- `num_successful_repair_steps`: `{trace['num_successful_repair_steps']}`\n")
        md.write(f"- `applicable_actions`: `{trace['applicable_actions_json']}`\n")
        md.write(f"- `num_steps`: `{trace['num_steps']}`\n\n")

        md.write("### Problem\n\n")
        md.write(clean_text(trace["problem_statement"], MAX_FIELD_CHARS) + "\n\n")

        md.write("### Answers\n\n")
        md.write(f"- Gold answer: `{clean_text(trace['gold_answer'], 500)}`\n")
        md.write(f"- Agent final answer: `{clean_text(trace['final_answer'], 500)}`\n\n")

        md.write("### Manual Labeling Fields\n\n")
        md.write("- Human action: \n")
        md.write("- Human rationale: \n")
        md.write("- Confidence: \n\n")

        md.write("### Trace Steps\n\n")
        for step in step_dicts:
            md.write(format_step(step))
            md.write("\n\n")

print(f"Exported {len(trace_dicts)} traces")
print(f"Markdown: {MD_PATH}")
print(f"JSONL:    {JSONL_PATH}")

Exported 638 traces
Markdown: data/labeling_exports/failed_traces.md
JSONL:    data/labeling_exports/failed_traces.jsonl
